# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, trim, upper,left,substring,coalesce, lit
from pyspark.sql.types import StringType

#Read from Bronze

In [0]:
df_crm_prd_info = spark.table("workspace.bronze.crm_prd_info")

#Trim

In [0]:

df_crm_prd_info = df_crm_prd_info.withColumn("prd_line", trim(col("prd_line")))
df_crm_prd_info.limit(15).display()


#Normalization

In [0]:
df_crm_prd_info = (
    df_crm_prd_info
    .withColumn(
        "prd_line",
        when(col("prd_line") == "R", lit("Road"))
        .when(col("prd_line") == "S", lit("Clothes"))
        .when(col("prd_line") == "T", lit("Touring"))
        .when(col("prd_line") == "R", lit("Road"))
        .when(col("prd_line") == "M", lit("Mountain"))
        .otherwise( coalesce(col("prd_line"), lit("n/a")) )         
    )
    .withColumn(
        "prd_cost",
        coalesce(col("prd_cost"), lit(0))
    )
    .withColumn(
        "prd_end_dt",
        coalesce(col("prd_end_dt"), lit("2100-01-01").cast("date"))
    )
    
)

#Rename column

In [0]:
col_mapping = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
    }
for old_name, new_name in col_mapping.items():
    df_crm_prd_info = df_crm_prd_info.withColumnRenamed(old_name, new_name)

#Writing Silver Table

In [0]:
df_crm_prd_info.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_production")
